In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [33]:
import pandas as pd
import io

# Replace 'your_file_path.csv' with the actual path to your CSV file in Google Drive
# For example: '/content/drive/My Drive/my_data.csv'
try:
    file_path = '/content/drive/My Drive/CustomerFeedback/sentiment-analysis.csv' # CHANGE THIS TO YOUR FILE PATH

    # Read the file content as raw text
    with open(file_path, 'r', encoding='utf-8') as f:
        raw_content = f.read()

    # Split into lines, remove outer quotes if present, and join back
    # This handles cases where each line in the CSV is itself a single quoted string
    cleaned_lines = []
    for line in raw_content.splitlines():
        if line.startswith('"') and line.endswith('"'):
            cleaned_lines.append(line[1:-1]) # Remove outer quotes
        else:
            cleaned_lines.append(line)

    cleaned_content = '\n'.join(cleaned_lines)

    # Read the cleaned content using StringIO
    df = pd.read_csv(io.StringIO(cleaned_content), sep=",", quotechar='"', skipinitialspace=True)

    print(f"Successfully loaded data from {file_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Please ensure the file exists and the path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded data from /content/drive/My Drive/CustomerFeedback/sentiment-analysis.csv


,Text,Sentiment,Source,Date/Time,User ID,Location,Confidence Score
0,"I love this product!""""",Positive,Twitter,2023-06-15 09:23:14,@user123,New York,0.85
1,"The service was terrible.""""",Negative,Yelp Reviews,2023-06-15 11:45:32,user456,Los Angeles,0.65
2,"This movie is amazing!""""",Positive,IMDb,2023-06-15 14:10:22,moviefan789,London,0.92
3,I'm so disappointed with their customer suppor...,Negative,Online Forum,2023-06-15 17:35:11,forumuser1,Toronto,0.78
4,"Just had the best meal of my life!""""",Positive,TripAdvisor,2023-06-16 08:50:59,foodie22,Paris,0.88


In [34]:
df = df.drop(["Source","Date/Time","User ID","Location"],axis=1)

In [35]:
df.head()

,Text,Sentiment,Confidence Score
0,"I love this product!""""",Positive,0.85
1,"The service was terrible.""""",Negative,0.65
2,"This movie is amazing!""""",Positive,0.92
3,I'm so disappointed with their customer suppor...,Negative,0.78
4,"Just had the best meal of my life!""""",Positive,0.88


In [36]:
df.isnull().sum()

,0
Text,0
Sentiment,0
Confidence Score,0


In [37]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [38]:
df["Sentiment"] = le.fit_transform(df["Sentiment"])

In [39]:
df.dtypes

,0
Text,object
Sentiment,int64
Confidence Score,float64


In [40]:
import re
def remove_html(text):
  text = re.sub(r"<.*>","",text)
  return text
df["Text"] = df["Text"].apply(remove_html)

In [41]:
import re
def remove_urls(text):
  text = re.sub(r"http\S+","",text)
  return text
df["Text"] = df["Text"].apply(remove_urls)

In [42]:
import re
def remove_punct(text):
  text = re.sub(r"[^A-Za-z0-9\s]","",text)
  return text
df["Text"] = df["Text"].apply(remove_punct)

In [43]:
df.head()

,Text,Sentiment,Confidence Score
0,I love this product,1,0.85
1,The service was terrible,0,0.65
2,This movie is amazing,1,0.92
3,Im so disappointed with their customer support,0,0.78
4,Just had the best meal of my life,1,0.88


In [44]:
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [45]:
from nltk.tokenize import word_tokenize

text = "The company spent $30,000,000 last year."
tokens = word_tokenize(text)
print(tokens)


['The', 'company', 'spent', '$', '30,000,000', 'last', 'year', '.']


In [46]:
def tokenize_word(text):
  tokens = word_tokenize(text)
  return " ".join(tokens)
df["Text"] = df["Text"].apply(tokenize_word)

In [47]:
df.head()

,Text,Sentiment,Confidence Score
0,I love this product,1,0.85
1,The service was terrible,0,0.65
2,This movie is amazing,1,0.92
3,Im so disappointed with their customer support,0,0.78
4,Just had the best meal of my life,1,0.88


In [48]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')

# Sample text
text = "This is a sample sentence showing stopword removal."

# Get English stopwords and tokenize
stop_words = set(stopwords.words('english'))
tokens = word_tokenize(text.lower())

# Remove stopwords
filtered_tokens = [word for word in tokens if word not in stop_words]

print("Original:", tokens)
print("Filtered:", filtered_tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Original: ['this', 'is', 'a', 'sample', 'sentence', 'showing', 'stopword', 'removal', '.']
Filtered: ['sample', 'sentence', 'showing', 'stopword', 'removal', '.']


In [49]:
def remove_stop(text):
  stop_words = set(stopwords.words('english'))
  tokens = word_tokenize(text.lower())

  # Remove stopwords
  filtered_tokens = [word for word in tokens if word not in stop_words]
  return " ".join(filtered_tokens)
df["Text"] = df["Text"].apply(remove_stop)


In [50]:
df.head()

,Text,Sentiment,Confidence Score
0,love product,1,0.85
1,service terrible,0,0.65
2,movie amazing,1,0.92
3,im disappointed customer support,0,0.78
4,best meal life,1,0.88


In [51]:
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [52]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
text = "The cats were running faster than the dogs."
tokens = word_tokenize(text)
lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]

print(f"Original Text: {text}")
print(f"Lemmatized Words: {lemmatized_words}")

Original Text: The cats were running faster than the dogs.
Lemmatized Words: ['The', 'cat', 'were', 'running', 'faster', 'than', 'the', 'dog', '.']


In [53]:
def lemma(text):
  tokens = word_tokenize(text)
  lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]
  return " ".join(lemmatized_words)
df["Text"] = df["Text"].apply(lemma)

In [54]:
df.head()

,Text,Sentiment,Confidence Score
0,love product,1,0.85
1,service terrible,0,0.65
2,movie amazing,1,0.92
3,im disappointed customer support,0,0.78
4,best meal life,1,0.88


In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["Text"])

In [62]:
import pandas as pd

# Convert the sparse matrix X to a dense DataFrame
X_df = pd.DataFrame(X.toarray())

# Combine the TF-IDF features (X_df) with the 'Sentiment' column from df
# The 'Sentiment' column will be added as the last column
combined_features = pd.concat([X_df, df['Sentiment']], axis=1)

print("Shape of combined features (TF-IDF + Sentiment):")
print(combined_features.shape)
print("First 5 rows of combined features:")
display(combined_features.head())

Shape of combined features (TF-IDF + Sentiment):
(96, 227)
First 5 rows of combined features:


,0,1,2,3,4,5,6,7,8,9,...,217,218,219,220,221,222,223,224,225,Sentiment
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.721657,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [64]:
final_df = pd.concat([combined_features, df['Confidence Score']], axis=1)

In [65]:
final_df.head()

,0,1,2,3,4,5,6,7,8,9,...,218,219,220,221,222,223,224,225,Sentiment,Confidence Score
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.85
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.65
2,0.0,0.0,0.0,0.0,0.0,0.0,0.721657,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.92
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.78
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.88


In [70]:
X = final_df.drop("Confidence Score", axis=1)
y = final_df["Confidence Score"]

In [71]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42)

In [72]:
import torch
import torch.nn as nn

In [76]:
X_train_tensor = torch.tensor(X_train.to_numpy(),dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.to_numpy(),dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy(),dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.to_numpy(),dtype=torch.float32).view(-1,1)

In [79]:
from torch.utils.data import TensorDataset,DataLoader

In [80]:
train_data = TensorDataset(X_train_tensor,y_train_tensor)
test_data = TensorDataset(X_test_tensor,y_test_tensor)

In [81]:
train_loader = DataLoader(train_data,batch_size=32,shuffle=True)
test_loader = DataLoader(test_data,batch_size=32)

In [82]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):
    super().__init__()

    self.hidden_size= hidden_size
    self.num_layers = num_layers

    self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
    self.fc = nn.Linear(hidden_size,1)
  def forward(self,x):
    out,_ = self.rnn(x)
    outputs = self.fc(out[:,-1,:])
    return outputs

In [83]:
import torch.optim as optim
model = RNN(input_size = X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [85]:
epochs = 10

for epoch in range(epochs):
  model.train()

  for xb,yb in train_loader:
    optimizer.zero_grad()
    xb = xb.unsqueeze(1)
    outputs = model(xb)
    loss = criterion(outputs,yb)
    loss.backward()
    optimizer.step()
  print(f"Epoch : {epoch+1}/{epochs} and Loss : {loss.item()}")

Epoch : 1/10 and Loss : 0.5782671570777893
Epoch : 2/10 and Loss : 0.5547566413879395
Epoch : 3/10 and Loss : 0.3996064364910126
Epoch : 4/10 and Loss : 0.2922767102718353
Epoch : 5/10 and Loss : 0.18570101261138916
Epoch : 6/10 and Loss : 0.1386537104845047
Epoch : 7/10 and Loss : 0.07122854888439178
Epoch : 8/10 and Loss : 0.054350707679986954
Epoch : 9/10 and Loss : 0.009017227217555046
Epoch : 10/10 and Loss : 0.005758231971412897


In [86]:
model.eval()

with torch.no_grad():
  for xb,yb in test_loader:
    xb = xb.unsqueeze(1)
    outputs = model(xb)
    loss = criterion(outputs,yb)
  print(f"Training Loss : {loss}")


Training Loss : 0.008790872059762478
